# PD06 · PQRS completas: agregaciones de alta cardinalidad en Polars CPU y GPU
Curso de Postgrado: BigData, Especialización en Bases de datos · CyberColombia.

**Plantilla individual por desarrollar.** Copia este notebook a `soluciones/`
con el mismo nombre. Completa código, controles e interpretación; no se entrega
una solución ni resultados medidos. La GPU y su kernel se consideran verificados.

Compararás **Polars CPU frente a Polars GPU**, con todos los datos de tres
periodos de Supersalud, sin pandas para las consultas. Resolverás dos ejercicios:

1. Departamentos con más quejas según edad, patología, motivo y canal.
2. Evolución de esas mismas combinaciones mes a mes, en orden cronológico ascendente.

Las claves de texto y el número de grupos aumentan el trabajo de agregación y
ordenación. Debes medirlo: no se garantiza saturar la GPU ni obtener aceleración.
No copies filas para simular más datos. Registra la cardinalidad observada.


## 1. Preparar los tres CSV completos

Desde una terminal Ubuntu/WSL, en la raíz del repositorio y con el entorno activo:

```bash
python kit/00_datos.py --descargar pqrs-completos
python kit/00_datos.py --verificar pqrs-completos
```

Deben quedar en `kit/data/pqrs_completos/`:

| Corte | CSV | Reportes de referencia |
|---|---|---:|
| 2023-II | `PQRD_2023_II.csv` | 716.697 |
| 2024-I | `PQRD_2024_I.csv` | 781.601 |
| 2024-II | `PQRD_2024_II.csv` | 946.468 |

Total: **2.444.766 reportes**, aproximadamente 1,82 GB de CSV. El descargador
conserva archivos correctos y verifica los tres. No son muestras.
No ejecutes esos comandos dentro del reloj del benchmark.


In [1]:
from pathlib import Path
from time import perf_counter
from importlib import metadata
import json, platform, sys
import polars as pl
from polars.testing import assert_frame_equal
from IPython.display import display

REPOSITORIO = None  # Ejemplo: Path('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate')
if REPOSITORIO is None:
    REPOSITORIO = next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'kit/pqrs_fuentes.json').is_file()), None)
assert REPOSITORIO is not None, 'Indicar la ruta del repositorio'
REPOSITORIO = Path(REPOSITORIO).expanduser().resolve()
KIT = REPOSITORIO / 'kit'
CSV_DIR = KIT / 'data/pqrs_completos'
OUT = KIT / 'salidas/soluciones/PD06'
PARQUET_DIR = OUT / 'parquet'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
PERIODOS = ['2023_II', '2024_I', '2024_II']
ARCHIVOS_CSV = [CSV_DIR / f'PQRD_{periodo}.csv' for periodo in PERIODOS]
ARCHIVOS_PARQUET = [PARQUET_DIR / f'{periodo}.parquet' for periodo in PERIODOS]
FILAS_ESPERADAS = [716697, 781601, 946468]
TOTAL_ESPERADO = sum(FILAS_ESPERADAS)
REPETICIONES = 3
TOP_N = 10
DISPOSITIVO_CUDA = 0
assert all(p.is_file() for p in ARCHIVOS_CSV), 'Completar descarga y verificación'
assert REPETICIONES >= 3
print('Python:', sys.executable)
print('Polars:', pl.__version__)
print('Entrada:', CSV_DIR)
print('Resultados:', OUT)


Python: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv-gpu/bin/python
Polars: 1.35.2
Entrada: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/pqrs_completos
Resultados: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD06


## 2. Leer y convertir con Polars · desarrollo del estudiante

Completa la siguiente celda sin pandas ni funciones del taller que oculten la conversión:

1. Inspecciona cinco filas de cada CSV con `pl.read_csv`. Usa `;`, `,` y `;`,
   respectivamente. Conserva códigos y las 38 columnas originales como texto.
2. Usa `pl.scan_csv` y transforma con expresiones Polars. Añade:
   - `corte_archivo`: corte del CSV de origen.
   - `codigo_departamento`: desde `afec_cod_depto`, dos dígitos; marca inválidos.
   - `edad_anios`: desde `afec_edad`, numérica. Explicita el tratamiento de edades
     vacías, negativas, no numéricas o no finitas; conserva estos reportes en un grupo nulo.
   - `anio` y `mes_num`: números desde `periodo` y `mes`, no desde el nombre semestral.
3. Escribe un Parquet por corte con `sink_parquet`, Zstandard y streaming CPU.
4. Comprueba que están los tres, que conservan sus filas y el contrato de columnas.
5. Al reejecutar, comprueba los Parquet existentes antes de decidir reutilizarlos.

La preparación se hace con Polars CPU; la GPU se utilizará para las agregaciones.
No mezcles este tiempo con los de consulta. Guarda el tiempo de conversión aparte.


In [2]:
# TODO: inspeccionar los tres CSV y desarrollar aquí su conversión con Polars.
# Operaciones esperadas: read_csv, scan_csv, with_columns y sink_parquet.
# No llamar a un notebook resuelto ni usar pandas para esta preparación.
# ---------- 2.1 Inspección: cinco filas de cada CSV, todo como texto ----------
SEPARADORES = {'2023_II': ';', '2024_I': ',', '2024_II': ';'}

for periodo, csv in zip(PERIODOS, ARCHIVOS_CSV):
    muestra = pl.read_csv(csv, separator=SEPARADORES[periodo], n_rows=5,
                          infer_schema=False, encoding='utf8')
    muestra = muestra.rename(lambda c: c.lstrip('\ufeff'))   # quita BOM si viene pegado
    print(periodo, '→', muestra.width, 'columnas')
    assert muestra.width == 38, f'{periodo}: se esperaban 38 columnas'
    display(muestra.select('periodo', 'mes', 'afec_cod_depto', 'afec_edad',
                           'patologia_1', 'cod_motesp', 'motivo_especifico', 'pqr_canal'))

# ---------- 2.2 Expresiones de normalización ----------
depto_txt = pl.col('afec_cod_depto').str.strip_chars()
depto_valido = depto_txt.str.contains(r'^\d{1,2}$')          # solo 1 o 2 dígitos

edad_txt = pl.col('afec_edad').str.strip_chars().str.replace(',', '.')
edad_num = edad_txt.cast(pl.Float64, strict=False)          # vacío / texto → null
edad_valida = edad_num.is_not_null() & edad_num.is_finite() & (edad_num >= 0)

def preparar(periodo, csv):
    return (
        pl.scan_csv(csv, separator=SEPARADORES[periodo],
                    infer_schema=False, encoding='utf8')   # 38 columnas como texto
        .rename(lambda c: c.lstrip('\ufeff'))
        .with_columns(
            pl.lit(periodo).alias('corte_archivo'),
            pl.col('cod_motesp').alias('cod_motesp_original'),
            pl.col('cod_motesp').str.strip_chars().str.zfill(6),
            pl.col('patologia_1', 'motivo_especifico', 'pqr_canal').str.strip_chars(),
            pl.when(depto_valido).then(depto_txt.str.zfill(2))
              .otherwise(None).alias('codigo_departamento'),
            (~depto_valido.fill_null(False)).alias('depto_invalido'),
            pl.when(edad_valida).then(edad_num).otherwise(None).alias('edad_anios'),
            (~edad_valida.fill_null(False)).alias('edad_invalida'),
            pl.col('periodo').str.strip_chars().cast(pl.Int32, strict=False).alias('anio'),
            pl.col('mes').str.strip_chars().cast(pl.Int8, strict=False).alias('mes_num'),
        )
    )

# ---------- 2.3 Escribir / reutilizar Parquet (CPU, streaming, zstd) ----------
def parquet_valido(ruta, filas):
    if not ruta.is_file():
        return False
    try:
        lf = pl.scan_parquet(ruta)
        ok_cols = {'codigo_departamento', 'edad_anios', 'anio', 'mes_num',
                   'corte_archivo'} <= set(lf.collect_schema().names())
        return ok_cols and lf.select(pl.len()).collect().item() == filas
    except Exception:
        return False

tiempos_conversion = {}
for periodo, csv, destino, filas in zip(PERIODOS, ARCHIVOS_CSV,
                                        ARCHIVOS_PARQUET, FILAS_ESPERADAS):
    if parquet_valido(destino, filas):
        print(f'{periodo}: Parquet existente verificado, se reutiliza')
        tiempos_conversion[periodo] = None
        continue
    t0 = perf_counter()
    preparar(periodo, csv).sink_parquet(destino, compression='zstd',
                                        engine='streaming')
    tiempos_conversion[periodo] = round(perf_counter() - t0, 2)
    print(f'{periodo}: convertido en {tiempos_conversion[periodo]} s')

# ---------- 2.4 Resumen de calidad (no entra en el benchmark) ----------
resumen = pl.concat([
    pl.scan_parquet(p).select(
        pl.first('corte_archivo'),
        pl.len().alias('filas'),
        pl.col('depto_invalido').sum().alias('deptos_invalidos'),
        pl.col('edad_invalida').sum().alias('edades_invalidas'),
        pl.col('anio').null_count().alias('anio_nulo'),
        pl.col('mes_num').null_count().alias('mes_nulo'),
        pl.col('mes_num').min().alias('mes_min'),
        pl.col('mes_num').max().alias('mes_max'),
    ) for p in ARCHIVOS_PARQUET
]).collect()
display(resumen)
print('Tiempos de conversión (s):', tiempos_conversion)

2023_II → 38 columnas


periodo,mes,afec_cod_depto,afec_edad,patologia_1,cod_motesp,motivo_especifico,pqr_canal
str,str,str,str,str,str,str,str
"""2023""","""7""","""11""","""69""","""ENFERMEDADES CRÓNICAS NO TRASM…","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Personalizado"""
"""2023""","""7""","""11""","""35""","""ENFERMEDADES HUÉRFANAS""","""040402""","""FALLAS O IRREGULARIDADES EN EL…","""Web"""
"""2023""","""7""","""70""","""77""","""ENFERMEDADES CRÓNICAS NO TRASM…","""010301""","""FALTA DE OPORTUNIDAD EN LA AUT…","""Personalizado"""
"""2023""","""7""","""76""","""71""","""PROBLEMAS RELACIONADOS CON FAC…","""010104""","""FALTA DE OPORTUNIDAD EN LA ENT…","""Escrito"""
"""2023""","""7""","""08""","""48""","""PROBLEMAS RELACIONADOS CON FAC…","""010304""","""FALTA DE OPORTUNIDAD EN LA AUT…","""Personalizado"""


2024_I → 38 columnas


periodo,mes,afec_cod_depto,afec_edad,patologia_1,cod_motesp,motivo_especifico,pqr_canal
str,str,str,str,str,str,str,str
"""2024""","""6""","""5""","""73""","""ENFERMEDADES OSTEOARTICULARES""","""10304""","""FALTA DE OPORTUNIDAD EN LA AUT…","""Telefónico"""
"""2024""","""4""","""63""","""84""","""PROBLEMAS RELACIONADOS CON FAC…","""10204""","""NEGACIÓN PARA LA ENTREGA DE TE…","""Redes Sociales"""
"""2024""","""5""","""11""","""0""","""MATERNO INFANTIL""","""10103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Web"""
"""2024""","""6""","""11""","""62""","""ENFERMEDADES OSTEOARTICULARES""","""10301""","""FALTA DE OPORTUNIDAD EN LA AUT…","""Web"""
"""2024""","""6""","""66""","""39""","""CÁNCER ""","""10103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Personalizado"""


2024_II → 38 columnas


periodo,mes,afec_cod_depto,afec_edad,patologia_1,cod_motesp,motivo_especifico,pqr_canal
str,str,str,str,str,str,str,str
"""2024""","""7""","""05""","""71""","""ENFERMEDADES CRÓNICAS NO TRASM…","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Web"""
"""2024""","""7""","""11""","""61""","""PROBLEMAS RELACIONADOS CON FAC…","""010203""","""NEGACIÓN EN LA ATENCIÓN EN OTR…","""Web"""
"""2024""","""7""","""66""","""32""","""ENFERMEDADES CRÓNICAS NO TRANS…","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Web"""
"""2024""","""7""","""66""","""9""","""PROBLEMAS RELACIONADOS CON FAC…","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Web"""
"""2024""","""7""","""05""","""33""","""PROBLEMAS RELACIONADOS CON FAC…","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Telefónico"""


2023_II: Parquet existente verificado, se reutiliza
2024_I: Parquet existente verificado, se reutiliza
2024_II: Parquet existente verificado, se reutiliza


corte_archivo,filas,deptos_invalidos,edades_invalidas,anio_nulo,mes_nulo,mes_min,mes_max
str,u32,u32,u32,u32,u32,i8,i8
"""2023_II""",716697,0,0,0,0,7,12
"""2024_I""",781601,0,0,0,0,1,6
"""2024_II""",946468,0,0,0,0,7,12


Tiempos de conversión (s): {'2023_II': None, '2024_I': None, '2024_II': None}


In [3]:
# Control de preparación; ejecutar después de completar la celda anterior.
assert all(p.is_file() for p in ARCHIVOS_PARQUET), 'Faltan periodos convertidos'
filas_por_archivo = [pl.scan_parquet(p).select(pl.len()).collect().item()
                     for p in ARCHIVOS_PARQUET]
assert filas_por_archivo == FILAS_ESPERADAS
assert sum(filas_por_archivo) == TOTAL_ESPERADO
REQUERIDAS = ['codigo_departamento', 'edad_anios', 'patologia_1',
              'cod_motesp', 'motivo_especifico', 'pqr_canal', 'anio', 'mes_num']
for archivo in ARCHIVOS_PARQUET:
    esquema = pl.scan_parquet(archivo).collect_schema()
    assert set(REQUERIDAS) <= set(esquema.names())
    assert esquema['anio'].is_integer() and esquema['mes_num'].is_integer()
    assert esquema['edad_anios'].is_numeric()
print('Tres periodos conservados:', filas_por_archivo)


Tres periodos conservados: [716697, 781601, 946468]


In [4]:
display(pl.scan_parquet(ARCHIVOS_PARQUET).group_by('corte_archivo')
        .agg(pl.col('cod_motesp').str.len_chars().unique().alias('largos'))
        .sort('corte_archivo').collect())

corte_archivo,largos
str,list[u32]
"""2023_II""",[6]
"""2024_I""",[6]
"""2024_II""",[6]


## 3. Ejercicio 1 · Departamentos con más quejas por perfil

**Combina todos los periodos completos**, no solo una muestra. Define el perfil:

| Dimensión | Campo |
|---|---|
| Edad en años | `edad_anios`, construida desde `afec_edad` |
| Patología | `patologia_1` |
| Código del motivo específico | `cod_motesp` |
| Descripción del mismo motivo | `motivo_especifico` |
| Canal de entrada | `pqr_canal` |

No mezcles un código de motivo específico con la descripción de otro nivel.
Agrupa por departamento y las cinco dimensiones; cuenta filas como `quejas`.
Ordena por perfil, quejas descendentes y código de departamento ascendente
para resolver empates de forma reproducible. Conserva los grupos no informados.

**Producto:** una tabla detallada con todas las combinaciones y sus conteos;
y una vista de hasta `TOP_N` departamentos con más quejas **para cada perfil**.
No basta un único top nacional que ignore edad, patología, motivo y canal.
Puedes preparar ese recorte después de medir, indicándolo en el informe.

El plan medido debe incluir lectura Parquet, agrupación y ordenamiento.
Declara un `LazyFrame` llamado `consulta_e1`; no lo ejecutes todavía.
No presentes quejas como personas únicas ni como tasas por población.


In [5]:
# TODO: desarrollar las expresiones y la consulta LazyFrame del ejercicio 1.
# Deben verse select, group_by, agg y sort.
consulta_e1 = None
PERFIL = ['edad_anios', 'patologia_1', 'cod_motesp', 'motivo_especifico', 'pqr_canal']
CLAVES_E1 = ['codigo_departamento', *PERFIL]

consulta_e1 = (
    pl.scan_parquet(ARCHIVOS_PARQUET)                 # los 3 periodos juntos
    .select(CLAVES_E1)
    .group_by(CLAVES_E1)                              # nulos = grupo propio, no se pierden
    .agg(pl.len().alias('quejas'))
    .sort([*PERFIL, 'quejas', 'codigo_departamento'],
          descending=[False] * len(PERFIL) + [True, False],
          nulls_last=True)
)
consulta_e1

## 4. Ejercicio 2 · Evolución mensual de los mismos perfiles

Conserva departamento, edad, patología, código y descripción del motivo y canal.
Añade **año y mes numéricos** a la agrupación, cuenta las quejas y ordena el
resultado **primero por año y mes ascendentes**, después por las dimensiones
para que la salida sea reproducible. Enero, febrero y octubre deben quedar
como meses 1, 2 y 10, no como cadenas ordenadas alfabéticamente.

**Producto:** tabla `anio`, `mes_num`, departamento, las cinco dimensiones y
`quejas`, que permita seguir un mismo perfil mes a mes. Muestra dos perfiles
concretos y explica sus cambios. No concluyas que toda variación es tendencia.

Conserva las fechas inválidas en el control total y sepáralas al presentar la
cronología. No inventes meses ni rellenes silenciosamente ausencias con cero.
Declara el plan lazy como `consulta_e2`, sin ejecutarlo todavía.


In [6]:
# TODO: desarrollar el ejercicio 2: mismas dimensiones + anio y mes_num.
# La ordenación por anio y mes_num debe estar en el plan medido.
consulta_e2 = None
CLAVES_E2 = ['anio', 'mes_num', 'codigo_departamento', *PERFIL]

consulta_e2 = (
    pl.scan_parquet(ARCHIVOS_PARQUET)
    .select(CLAVES_E2)
    .group_by(CLAVES_E2)
    .agg(pl.len().alias('quejas'))
    .sort(CLAVES_E2, nulls_last=True)        # año y mes numéricos ascendentes primero
)
consulta_e2

## 5. Contrato de comparación y frontera temporal

Completa las listas de claves y la comprobación para cada resultado.
Verifica todas las filas agregadas, no solo la suma global. Los conteos deben
ser exactos y cada consulta debe conservar 2.444.766 reportes, incluidos grupos
no informados. Revisa tipos, nulos y duplicados de claves agregadas.

Mediremos con `perf_counter`: **leer Parquet + agregar + ordenar + devolver un
DataFrame Polars en CPU**. `collect` termina esa frontera para ambos motores.
No se mide el tiempo de construir el plan, descargar, convertir, verificar ni
exportar. La medición GPU incluye transferencias; no es tiempo exclusivo de
kernels CUDA ni una comparación de datos ya residentes en VRAM.

CPU usará `engine='streaming'`. GPU usará `GPUEngine(raise_on_fail=True)`.
Si el plan no está soportado en GPU, corrígelo o informa el error; no lo midas
como GPU mediante fallback a CPU. No cambies filtros o claves entre motores.


In [7]:
assert isinstance(consulta_e1, pl.LazyFrame), 'Completar ejercicio 1'
assert isinstance(consulta_e2, pl.LazyFrame), 'Completar ejercicio 2'
CONSULTAS = {'e1_departamentos': consulta_e1, 'e2_evolucion_mensual': consulta_e2}
CLAVES_RESULTADO = {
    'e1_departamentos': CLAVES_E1,
    'e2_evolucion_mensual': CLAVES_E2,
}
assert all(CLAVES_RESULTADO.values()), 'Definir las claves para comparar todos los grupos'
engine_gpu = pl.GPUEngine(device=DISPOSITIVO_CUDA, raise_on_fail=True)

# Esquema esperado del plan (no ejecuta la consulta)
for nombre, plan in CONSULTAS.items():
    esquema = plan.collect_schema()
    assert esquema.names() == [*CLAVES_RESULTADO[nombre], 'quejas'], nombre
    assert esquema['quejas'].is_integer()
    print(nombre, dict(esquema))

def validar(df, nombre):
    """Controles de un resultado ya calculado (fuera del reloj)."""
    claves = CLAVES_RESULTADO[nombre]
    assert df.columns == [*claves, 'quejas']
    assert not df.select(claves).is_duplicated().any(), f'{nombre}: claves duplicadas'
    assert df['quejas'].null_count() == 0 and (df['quejas'] > 0).all()
    assert df['quejas'].sum() == TOTAL_ESPERADO, f'{nombre}: se perdieron reportes'
    nulos = {c: df.filter(pl.col(c).is_null())['quejas'].sum() for c in claves}
    return {'grupos': df.height, 'reportes': int(df['quejas'].sum()),
            'reportes_con_clave_nula': {c: int(v) for c, v in nulos.items() if v}}

primeras = []

e1_departamentos {'codigo_departamento': String, 'edad_anios': Float64, 'patologia_1': String, 'cod_motesp': String, 'motivo_especifico': String, 'pqr_canal': String, 'quejas': UInt32}
e2_evolucion_mensual {'anio': Int32, 'mes_num': Int8, 'codigo_departamento': String, 'edad_anios': Float64, 'patologia_1': String, 'cod_motesp': String, 'motivo_especifico': String, 'pqr_canal': String, 'quejas': UInt32}


### Ejercicio 1 · Primera ejecución en CPU y GPU

Ejecuta la celda después de completar los planes. La primera llamada puede
incluir inicialización y cachés; se guarda por separado de las repeticiones.


In [8]:
# CPU: se cronometra la ejecución completa del plan, no su declaración.
inicio = perf_counter()
cpu_e1 = consulta_e1.collect(engine='streaming')
tiempo_cpu_e1 = perf_counter() - inicio

# GPU: mismos datos, claves, agregación y orden.
inicio = perf_counter()
gpu_e1 = consulta_e1.collect(engine=engine_gpu)
tiempo_gpu_e1 = perf_counter() - inicio

# Fuera del reloj: comparar todos los grupos y conservar el total.
assert_frame_equal(cpu_e1.sort(CLAVES_RESULTADO['e1_departamentos']),
                   gpu_e1.sort(CLAVES_RESULTADO['e1_departamentos']), check_exact=True)
assert cpu_e1['quejas'].sum() == TOTAL_ESPERADO
primeras.extend([
    {'ejercicio': 'e1', 'motor': 'cpu', 'segundos': tiempo_cpu_e1},
    {'ejercicio': 'e1', 'motor': 'gpu', 'segundos': tiempo_gpu_e1}])
print('Ejercicio 1 · CPU:', tiempo_cpu_e1, 's · GPU:', tiempo_gpu_e1, 's')
display(gpu_e1.head(20))


Ejercicio 1 · CPU: 0.660152098999987 s · GPU: 10.733079523000015 s


codigo_departamento,edad_anios,patologia_1,cod_motesp,motivo_especifico,pqr_canal,quejas
str,f64,str,str,str,str,u32
"""76""",0.0,"""COVID -19""","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Chat""",1
"""11""",0.0,"""COVID -19""","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Web""",1
"""08""",0.0,"""COVID -19""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Personalizado""",1
"""05""",0.0,"""COVID -19""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Telefónico""",1
"""11""",0.0,"""COVID -19""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Web""",1
…,…,…,…,…,…,…
"""11""",0.0,"""COVID -19""","""020101""","""INSATISFACCIÓN POR FALTA DE OP…","""Web""",1
"""11""",0.0,"""COVID -19""","""020201""","""INSATISFACCIÓN RELACIONADA CON…","""Escrito""",1
"""05""",0.0,"""CUIDADO INTENSIVO PARA CUALQUI…","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Personalizado""",1


### Ejercicio 2 · Primera ejecución en CPU y GPU

Ejecuta la celda después de completar los planes. La primera llamada puede
incluir inicialización y cachés; se guarda por separado de las repeticiones.


In [9]:
# CPU: se cronometra la ejecución completa del plan, no su declaración.
inicio = perf_counter()
cpu_e2 = consulta_e2.collect(engine='streaming')
tiempo_cpu_e2 = perf_counter() - inicio

# GPU: mismos datos, claves, agregación y orden.
inicio = perf_counter()
gpu_e2 = consulta_e2.collect(engine=engine_gpu)
tiempo_gpu_e2 = perf_counter() - inicio

# Fuera del reloj: comparar todos los grupos y conservar el total.
assert_frame_equal(cpu_e2.sort(CLAVES_RESULTADO['e2_evolucion_mensual']),
                   gpu_e2.sort(CLAVES_RESULTADO['e2_evolucion_mensual']), check_exact=True)
assert cpu_e2['quejas'].sum() == TOTAL_ESPERADO
primeras.extend([
    {'ejercicio': 'e2', 'motor': 'cpu', 'segundos': tiempo_cpu_e2},
    {'ejercicio': 'e2', 'motor': 'gpu', 'segundos': tiempo_gpu_e2}])
print('Ejercicio 2 · CPU:', tiempo_cpu_e2, 's · GPU:', tiempo_gpu_e2, 's')
display(gpu_e2.head(20))


Ejercicio 2 · CPU: 1.0423937980000062 s · GPU: 1.1187473139999895 s


anio,mes_num,codigo_departamento,edad_anios,patologia_1,cod_motesp,motivo_especifico,pqr_canal,quejas
i32,i8,str,f64,str,str,str,str,u32
2023,7,"""05""",0.0,"""COVID -19""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Telefónico""",1
2023,7,"""05""",0.0,"""ENFERMEDADES CRÓNICAS NO TRASM…","""010302""","""FALTA DE OPORTUNIDAD EN LA AUT…","""Telefónico""",1
2023,7,"""05""",0.0,"""ENFERMEDADES CRÓNICAS NO TRASM…","""010304""","""FALTA DE OPORTUNIDAD EN LA AUT…","""Web""",1
2023,7,"""05""",0.0,"""ENFERMEDADES HUÉRFANAS""","""010101""","""FALTA DE OPORTUNIDAD EN LAS CI…","""Personalizado""",1
2023,7,"""05""",0.0,"""ENFERMEDADES HUÉRFANAS""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Chat""",1
…,…,…,…,…,…,…,…,…
2023,7,"""05""",0.0,"""MATERNO INFANTIL""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Telefónico""",8
2023,7,"""05""",0.0,"""MATERNO INFANTIL""","""010103""","""FALTA DE OPORTUNIDAD EN LA ATE…","""Web""",11
2023,7,"""05""",0.0,"""MATERNO INFANTIL""","""010104""","""FALTA DE OPORTUNIDAD EN LA ENT…","""Telefónico""",3


## 6. Calentamiento y tres repeticiones por motor

Completa el bucle de medición. Por cada ejercicio:

1. Ejecuta una vuelta adicional de calentamiento CPU/GPU y no la mezcles con las medidas.
2. Repite al menos tres veces, alternando CPU→GPU y GPU→CPU.
3. Rodea cada `collect` con `perf_counter`; compara resultados después de parar el reloj.
4. Guarda `ejercicio`, `motor`, `repeticion`, `segundos`, `filas_entrada`,
   `grupos_resultado` y `equivalente` en la lista `mediciones`.
5. Revisa que los resultados conserven todos los reportes y las claves esperadas.

Mide los mismos planes usados arriba. Mantén las mismas condiciones y registra
hilos CPU, GPU, VRAM, controlador y versiones. No se borran automáticamente las
cachés del sistema operativo; explica su efecto. No uses `process_time` como
sustituto de la duración total de la consulta GPU.


In [10]:
import os, gc
mediciones = []

# Resultados de referencia (de las primeras ejecuciones en CPU)
REFERENCIAS = {'e1_departamentos': cpu_e1, 'e2_evolucion_mensual': cpu_e2}
for nombre in CONSULTAS:
    print(nombre, validar(REFERENCIAS[nombre], nombre), flush=True)

# Liberar copias GPU que ya no se usan
for v in ['gpu_e1', 'gpu_e2']:
    globals().pop(v, None)
gc.collect()

# GPU por bloques para usar menos VRAM (6 GB)
engine_gpu = pl.GPUEngine(device=DISPOSITIVO_CUDA, raise_on_fail=True,
                          executor='streaming')

def equivalente(df, nombre):
    claves = CLAVES_RESULTADO[nombre]
    try:
        assert_frame_equal(df.sort(claves, nulls_last=True),
                           REFERENCIAS[nombre].sort(claves, nulls_last=True),
                           check_exact=True)
        return True
    except AssertionError:
        return False

MOTORES = {'cpu': 'streaming', 'gpu': engine_gpu}

for nombre, plan in CONSULTAS.items():
    # Calentamiento: una vuelta por motor, no se registra
    for motor, engine in MOTORES.items():
        print(f'{nombre} calentamiento {motor}...', flush=True)
        _ = plan.collect(engine=engine)
        del _
        gc.collect()

    for rep in range(1, REPETICIONES + 1):
        orden = ['cpu', 'gpu'] if rep % 2 == 1 else ['gpu', 'cpu']   # alternar
        for motor in orden:
            print(f'{nombre} rep{rep} {motor} ejecutando...', flush=True)
            inicio = perf_counter()
            resultado = plan.collect(engine=MOTORES[motor])        # lo medido
            segundos = perf_counter() - inicio
            # Fuera del reloj: validar y registrar
            mediciones.append({
                'ejercicio': nombre, 'motor': motor, 'repeticion': rep,
                'segundos': segundos,
                'filas_entrada': int(resultado['quejas'].sum()),
                'grupos_resultado': resultado.height,
                'equivalente': equivalente(resultado, nombre),
            })
            print(f'   → {segundos:.3f} s · equivalente={mediciones[-1]["equivalente"]}',
                  flush=True)
            del resultado
            gc.collect()

display(pl.DataFrame(mediciones))
print('Hilos Polars CPU:', pl.thread_pool_size(), '· núcleos lógicos:', os.cpu_count())

e1_departamentos {'grupos': 571235, 'reportes': 2444766, 'reportes_con_clave_nula': {}}
e2_evolucion_mensual {'grupos': 1492458, 'reportes': 2444766, 'reportes_con_clave_nula': {}}
e1_departamentos calentamiento cpu...
e1_departamentos calentamiento gpu...
e1_departamentos rep1 cpu ejecutando...
   → 0.560 s · equivalente=True
e1_departamentos rep1 gpu ejecutando...
   → 0.687 s · equivalente=True
e1_departamentos rep2 gpu ejecutando...
   → 0.816 s · equivalente=True
e1_departamentos rep2 cpu ejecutando...
   → 0.544 s · equivalente=True
e1_departamentos rep3 cpu ejecutando...
   → 0.547 s · equivalente=True
e1_departamentos rep3 gpu ejecutando...
   → 1.062 s · equivalente=True
e2_evolucion_mensual calentamiento cpu...
e2_evolucion_mensual calentamiento gpu...
e2_evolucion_mensual rep1 cpu ejecutando...
   → 0.968 s · equivalente=True
e2_evolucion_mensual rep1 gpu ejecutando...
   → 1.234 s · equivalente=True
e2_evolucion_mensual rep2 gpu ejecutando...
   → 1.001 s · equivalente=True

ejercicio,motor,repeticion,segundos,filas_entrada,grupos_resultado,equivalente
str,str,i64,f64,i64,i64,bool
"""e1_departamentos""","""cpu""",1,0.560083,2444766,571235,true
"""e1_departamentos""","""gpu""",1,0.686768,2444766,571235,true
"""e1_departamentos""","""gpu""",2,0.816297,2444766,571235,true
"""e1_departamentos""","""cpu""",2,0.54354,2444766,571235,true
"""e1_departamentos""","""cpu""",3,0.546718,2444766,571235,true
…,…,…,…,…,…,…
"""e2_evolucion_mensual""","""gpu""",1,1.234224,2444766,1492458,true
"""e2_evolucion_mensual""","""gpu""",2,1.001405,2444766,1492458,true
"""e2_evolucion_mensual""","""cpu""",2,1.002738,2444766,1492458,true


Hilos Polars CPU: 24 · núcleos lógicos: 24


## 7. Tabla de tiempos y factor de aceleración

La celda siguiente no resuelve consultas: resume tus mediciones. Habrá
**12 filas como mínimo**: dos ejercicios × dos motores × tres repeticiones.
Factor CPU/GPU = mediana CPU / mediana GPU: mayor que 1 favorece GPU;
menor que 1 favorece CPU. No mezcles ambas consultas ni las primeras ejecuciones.


In [11]:
assert len(mediciones) == len(CONSULTAS) * 2 * REPETICIONES
requeridas = {'ejercicio', 'motor', 'repeticion', 'segundos',
              'filas_entrada', 'grupos_resultado', 'equivalente'}
assert all(requeridas <= set(fila) for fila in mediciones)
tiempos = pl.DataFrame(mediciones)
assert tiempos['equivalente'].all()
assert (tiempos['filas_entrada'] == TOTAL_ESPERADO).all()
assert tiempos['segundos'].is_finite().all() and (tiempos['segundos'] > 0).all()
assert set(tiempos['ejercicio'].to_list()) == set(CONSULTAS)
assert set(tiempos['motor'].to_list()) == {'cpu', 'gpu'}
conteos = tiempos.group_by(['ejercicio', 'motor']).agg(pl.len().alias('n'))
assert (conteos['n'] == REPETICIONES).all()
assert tiempos.select(['ejercicio', 'motor', 'repeticion']).unique().height == tiempos.height
resumen = tiempos.group_by(['ejercicio', 'motor']).agg(
    pl.col('segundos').median().alias('mediana_s'),
    pl.col('segundos').min().alias('min_s'),
    pl.col('segundos').max().alias('max_s')).sort(['ejercicio', 'motor'])
factores = []
for nombre in CONSULTAS:
    cpu_s = resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == 'cpu'))['mediana_s'].item()
    gpu_s = resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == 'gpu'))['mediana_s'].item()
    factores.append({'ejercicio': nombre, 'mediana_cpu_s': cpu_s,
                     'mediana_gpu_s': gpu_s, 'factor_cpu_sobre_gpu': cpu_s / gpu_s})
display(resumen)
display(pl.DataFrame(factores))
tiempos.write_csv(OUT / 'benchmark_PD06.csv')
resumen.write_csv(OUT / 'resumen_PD06.csv')
pl.DataFrame(primeras).write_csv(OUT / 'primeras_ejecuciones_PD06.csv')
pl.DataFrame(factores).write_csv(OUT / 'factores_PD06.csv')


ejercicio,motor,mediana_s,min_s,max_s
str,str,f64,f64,f64
"""e1_departamentos""","""cpu""",0.546718,0.54354,0.560083
"""e1_departamentos""","""gpu""",0.816297,0.686768,1.062026
"""e2_evolucion_mensual""","""cpu""",0.972324,0.967766,1.002738
"""e2_evolucion_mensual""","""gpu""",1.234224,1.001405,1.24217


ejercicio,mediana_cpu_s,mediana_gpu_s,factor_cpu_sobre_gpu
str,f64,f64,f64
"""e1_departamentos""",0.546718,0.816297,0.669753
"""e2_evolucion_mensual""",0.972324,1.234224,0.787802


## 8. Interpretar y entregar

Completa el notebook y entrega en tu repositorio:

- `soluciones/PD06_PQRS_agregaciones_CPU_GPU.ipynb` con código visible y conclusiones.
- Resultados regenerables en `kit/salidas/soluciones/PD06/`: detalle/ranking de
  departamentos, evolución mensual, controles, tiempos, resumen y factores.
- Un control JSON con filas por corte, total conciliado, número de grupos por
  consulta, reglas de edad y fechas, cobertura de departamentos, versiones,
  modelo GPU/VRAM/controlador, hilos CPU y alcance de cada medición.

**Responde:** ¿qué departamentos lideran dentro de cada perfil? ¿Cómo evolucionan
los dos perfiles seleccionados? ¿Cuántos grupos genera cada consulta? ¿La GPU
acelera ambas? ¿Qué parte puede estar limitada por lectura, transferencias,
ordenamiento o memoria? ¿Cambió la conclusión entre primera ejecución y mediana?

Se evalúan corrección, equivalencia, medición reproducible e interpretación;
no que GPU necesariamente gane. No adjuntes los CSV originales al repositorio.
La salida queda en la ruta ignorada `kit/salidas/`. La copia resuelta pertenece
al estudiante; esta plantilla mantiene sus celdas por completar.

Apoyo: [guía Polars](../docs/manual-polars.md),
[entorno GPU](../docs/polars-gpu.md),
[referencia P04](../talleres/P04.md). Son referencias, no una solución de PD06.


In [13]:
top_e1 = (
    cpu_e1
    .with_columns((pl.int_range(pl.len()).over(PERFIL) + 1).alias('posicion'))
    .filter(pl.col('posicion') <= TOP_N)
)
print('Grupos totales:', cpu_e1.height, '· filas en el top:', top_e1.height)
display(top_e1.sort('quejas', descending=True).head(15))

import subprocess
# Productos regenerables
top_e1.write_csv(OUT / 'e1_top_departamentos_por_perfil.csv')
cpu_e1.write_parquet(OUT / 'e1_detalle_departamentos.parquet', compression='zstd')
cpu_e2.write_parquet(OUT / 'e2_evolucion_mensual.parquet', compression='zstd')

# Dos perfiles concretos: los dos con más quejas y todas sus claves informadas
perfiles = (cpu_e1.drop_nulls(CLAVES_E1)
            .sort('quejas', descending=True).head(2).select(CLAVES_E1))
evolucion = (cpu_e2.join(perfiles, on=CLAVES_E1, how='semi')
             .filter(pl.col('anio').is_not_null() & pl.col('mes_num').is_not_null())
             .sort([*CLAVES_E1, 'anio', 'mes_num']))
display(perfiles)
display(evolucion)
evolucion.write_csv(OUT / 'e2_dos_perfiles.csv')
fechas_invalidas = cpu_e2.filter(pl.col('anio').is_null() | pl.col('mes_num').is_null())['quejas'].sum()
print('Reportes con fecha inválida (fuera de la cronología):', fechas_invalidas)

def cmd(args):
    try:
        return subprocess.run(args, capture_output=True, text=True, check=True).stdout.strip()
    except Exception as e:
        return f'no disponible: {e}'

try:
    import cudf_polars; version_cudf = cudf_polars.__version__
except Exception as e:
    version_cudf = f'no disponible: {e}'

control = {
    'filas_por_corte': dict(zip(PERIODOS, filas_por_archivo)),
    'total_conciliado': int(sum(filas_por_archivo)),
    'grupos': {n: validar(REFERENCIAS[n], n) for n in CONSULTAS},
    'reglas': {
        'edad': 'afec_edad → Float64; vacía, no numérica, no finita o negativa → null (grupo nulo, se conserva)',
        'fechas': 'anio desde periodo, mes_num desde mes, enteros; nulos se conservan y se excluyen solo de la cronología',
        'departamento': 'afec_cod_depto de 1-2 dígitos → 2 dígitos con cero; otro valor → null',
        'cod_motesp': 'normalizado a 6 dígitos (2024_I venía sin cero inicial); original en cod_motesp_original',
    },
    'cobertura_departamentos': cpu_e1['codigo_departamento'].drop_nulls().n_unique(),
    'fechas_invalidas': int(fechas_invalidas),
    'versiones': {'python': platform.python_version(), 'polars': pl.__version__,
                  'cudf_polars': version_cudf},
    'gpu': cmd(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader']),
    'hilos_cpu_polars': pl.thread_pool_size(),
    'alcance_medicion': 'perf_counter alrededor de collect: leer Parquet + agrupar + ordenar + DataFrame en CPU; GPU incluye transferencias; sin conversión ni exportación',
    'repeticiones': REPETICIONES,
}
(OUT / 'control_PD06.json').write_text(json.dumps(control, indent=2, ensure_ascii=False))
print(json.dumps(control, indent=2, ensure_ascii=False))

Grupos totales: 571235 · filas en el top: 449119


codigo_departamento,edad_anios,patologia_1,cod_motesp,motivo_especifico,pqr_canal,quejas,posicion
str,f64,str,str,str,str,u32,i64
"""11""",0.0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",886,1
"""11""",63.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",803,1
"""11""",68.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",742,1
"""11""",67.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",741,1
"""11""",1.0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",737,1
…,…,…,…,…,…,…,…
"""11""",69.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",702,1
"""11""",61.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",700,1
"""11""",66.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",692,1


codigo_departamento,edad_anios,patologia_1,cod_motesp,motivo_especifico,pqr_canal
str,f64,str,str,str,str
"""11""",0.0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web"""
"""11""",63.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web"""


anio,mes_num,codigo_departamento,edad_anios,patologia_1,cod_motesp,motivo_especifico,pqr_canal,quejas
i32,i8,str,f64,str,str,str,str,u32
2023,7,"""11""",0.0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",28
2023,8,"""11""",0.0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",66
2023,9,"""11""",0.0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",50
2023,10,"""11""",0.0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",58
2023,11,"""11""",0.0,"""MATERNO INFANTIL""","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",71
…,…,…,…,…,…,…,…,…
2024,8,"""11""",63.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",42
2024,9,"""11""",63.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",48
2024,10,"""11""",63.0,"""PROBLEMAS RELACIONADOS CON FAC…","""010201""","""NEGACIÓN EN LA ASIGNACIÓN DE C…","""Web""",54


Reportes con fecha inválida (fuera de la cronología): 0
{
  "filas_por_corte": {
    "2023_II": 716697,
    "2024_I": 781601,
    "2024_II": 946468
  },
  "total_conciliado": 2444766,
  "grupos": {
    "e1_departamentos": {
      "grupos": 571235,
      "reportes": 2444766,
      "reportes_con_clave_nula": {}
    },
    "e2_evolucion_mensual": {
      "grupos": 1492458,
      "reportes": 2444766,
      "reportes_con_clave_nula": {}
    }
  },
  "reglas": {
    "edad": "afec_edad → Float64; vacía, no numérica, no finita o negativa → null (grupo nulo, se conserva)",
    "fechas": "anio desde periodo, mes_num desde mes, enteros; nulos se conservan y se excluyen solo de la cronología",
    "departamento": "afec_cod_depto de 1-2 dígitos → 2 dígitos con cero; otro valor → null",
    "cod_motesp": "normalizado a 6 dígitos (2024_I venía sin cero inicial); original en cod_motesp_original"
  },
  "cobertura_departamentos": 33,
  "fechas_invalidas": 0,
  "versiones": {
    "python": "3.12.15",
  

## Conclusiones

**Preparación.** Convertí los tres cortes a Parquet y se conservaron los 2.444.766 reportes (716.697 + 781.601 + 946.468). En 2024-I el `cod_motesp` venía sin el cero inicial, así que lo normalicé a 6 dígitos para no partir el mismo motivo en dos grupos. No hubo edades, fechas ni departamentos inválidos, y aparecen los 33 departamentos.

**Departamentos que lideran.** Bogotá (11) queda de primera en los perfiles con más quejas. Casi todos esos perfiles son por el motivo 010201 (negación en la asignación de citas) y llegan por la web. El perfil más grande es de bebés de 0 años en materno infantil (886 quejas).

**Evolución de los dos perfiles.** Seguí esos dos perfiles mes a mes. El de bebés arranca con 28 quejas en julio de 2023 y sube a entre 50 y 71 en los meses siguientes. El de 63 años se mueve entre 41 y 54 en el segundo semestre de 2024 y cae a 26 en diciembre. Con 18 meses no lo tomaría como tendencia; la caída de diciembre puede ser por la temporada.

**Grupos.** El ejercicio 1 genera 571.235 grupos y el ejercicio 2, 1.492.458. En el segundo hay en promedio 1,6 quejas por grupo, o sea que la agregación casi no reduce los datos.

**¿Acelera la GPU?** No. Con la mediana, la CPU fue más rápida en los dos casos: 0,55 s contra 0,82 s en el ejercicio 1 (factor 0,67) y 0,97 s contra 1,23 s en el ejercicio 2 (factor 0,79). Los resultados fueron idénticos en todas las repeticiones, pero la GPU varió más (de 0,69 a 1,06 s).

**Qué lo limita.** Creo que el tiempo se va más en mover datos que en calcular:
- los Parquet se leen desde `/mnt/c` (disco de Windows visto desde WSL), que es lento;
- los datos tienen que subir a la GPU y el resultado, con hasta 1,5 millones de filas de texto, tiene que volver a la CPU;
- ordenar tantas columnas de texto pesa;
- con 6 GB de VRAM usé `executor='streaming'`, que procesa por bloques.

Además, Polars usó 24 hilos de CPU, así que la CPU era una rival fuerte. Con 2,4 millones de filas no hay suficiente volumen para que la GPU compense esos costos.

**Primera ejecución vs mediana.** En la primera corrida del ejercicio 1 la GPU tardó 10,7 s, contra 0,66 s de la CPU. Casi todo ese tiempo fue la inicialización de CUDA, porque en las repeticiones bajó a menos de 1 s. En el ejercicio 2, que corrió con la GPU ya inicializada, quedaron casi parejas (1,04 s CPU vs 1,12 s GPU). La conclusión no cambió: la CPU ganó en la primera ejecución y en la mediana. Lo que sí cambió fue el tamaño de la diferencia, y por eso es importante calentar antes de medir.

**Problemas en el camino.** El kernel se reiniciaba en el ejercicio 2 porque WSL solo tenía 7,6 GB de RAM. Lo subí a 12 GB con `.wslconfig` y cerré un Jupyter viejo y un Spark que seguían corriendo.